# 🖼️ Notebook 1 — Exploration du Dataset DIV2K

## Restauration d'images : Débruitage et Super-Résolution

---

### Objectif de ce notebook

Ce notebook a pour but d'explorer et de comprendre le dataset **DIV2K** (DIVerse 2K resolution images) avant toute phase d'entraînement.  
Une bonne compréhension des données est fondamentale en apprentissage automatique : elle permet de détecter les biais, d'évaluer la qualité du dataset et d'adapter les stratégies de prétraitement.

**Plan du notebook :**
1. Configuration de l'environnement
2. Chargement et inspection du dataset
3. Analyse des résolutions
4. Analyse des distributions de pixels
5. Analyse des canaux couleur
6. Détection d'images corrompues
7. Visualisation PCA de la diversité
8. Conclusions et bilan

## 1. Configuration de l'environnement

On commence par importer toutes les bibliothèques nécessaires et définir les chemins vers les données.  
Toutes les figures générées seront sauvegardées dans le dossier `results/data_exploration`.

In [ ]:
import os
import glob
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from pathlib import Path
from collections import Counter
from tqdm.notebook import tqdm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ── Chemins ──────────────────────────────────────────────────────────────────
DATASET_PATH = "/kaggle/input/datasets/soumikrakshit/div2k-high-resolution-images/DIV2K_train_HR/DIV2K_train_HR"
RESULTS_DIR  = Path("results/data_exploration")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Seed pour la reproductibilité ────────────────────────────────────────────
random.seed(42)
np.random.seed(42)

print(f"✅ Environnement configuré.")
print(f"📁 Dossier de résultats : {RESULTS_DIR.resolve()}")

## 2. Chargement et inspection du dataset

DIV2K est un dataset de référence pour la super-résolution. Il contient **800 images d'entraînement** à haute résolution (2K), représentant une grande diversité de scènes.  
On commence par lister les fichiers, inspecter leur format et afficher quelques statistiques de base.

In [ ]:
# Lister toutes les images
extensions = ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tif', '*.tiff']
all_images = []
for ext in extensions:
    all_images.extend(glob.glob(os.path.join(DATASET_PATH, ext)))
all_images = sorted(all_images)

print(f"📦 Nombre total d'images trouvées : {len(all_images)}")
print(f"\n📋 Exemples de fichiers :")
for f in all_images[:5]:
    print(f"   {os.path.basename(f)}")

# Statistiques sur les formats
formats = Counter([os.path.splitext(f)[1].lower() for f in all_images])
print(f"\n🗂️  Formats détectés : {dict(formats)}")

In [ ]:
# Extraire les métadonnées de chaque image
metadata = []
corrupted = []

print("🔍 Analyse des métadonnées en cours...")
for img_path in tqdm(all_images):
    try:
        with Image.open(img_path) as img:
            w, h = img.size
            mode = img.mode
            file_size = os.path.getsize(img_path) / 1024  # KB
            metadata.append({
                'filename': os.path.basename(img_path),
                'width': w,
                'height': h,
                'pixels': w * h,
                'aspect_ratio': round(w / h, 3),
                'mode': mode,
                'file_size_kb': round(file_size, 1)
            })
    except Exception as e:
        corrupted.append({'filename': os.path.basename(img_path), 'error': str(e)})

df = pd.DataFrame(metadata)
print(f"\n✅ {len(df)} images analysées avec succès.")
print(f"❌ {len(corrupted)} image(s) corrompue(s) détectée(s).")
df.head()

In [ ]:
# Statistiques descriptives du dataset
print("📊 Statistiques descriptives du dataset DIV2K :")
print(df[['width', 'height', 'pixels', 'aspect_ratio', 'file_size_kb']].describe().round(2))

## 3. Analyse des résolutions

La résolution des images est un facteur clé pour la super-résolution. On souhaite comprendre :  
- La distribution des largeurs et hauteurs,  
- Les ratios d'aspect (paysage vs portrait),  
- Si les images sont suffisamment grandes pour en extraire des patches de haute qualité.

Des images trop petites ou trop homogènes en résolution pourraient limiter la généralisation du modèle.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Analyse des résolutions — Dataset DIV2K', fontsize=15, fontweight='bold', y=1.01)

# Distribution des largeurs
axes[0, 0].hist(df['width'], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
axes[0, 0].set_title('Distribution des largeurs (pixels)')
axes[0, 0].set_xlabel('Largeur (px)')
axes[0, 0].set_ylabel('Nombre d\'images')
axes[0, 0].axvline(df['width'].mean(), color='red', linestyle='--', label=f'Moyenne: {df["width"].mean():.0f}px')
axes[0, 0].legend()

# Distribution des hauteurs
axes[0, 1].hist(df['height'], bins=30, color='seagreen', edgecolor='white', alpha=0.85)
axes[0, 1].set_title('Distribution des hauteurs (pixels)')
axes[0, 1].set_xlabel('Hauteur (px)')
axes[0, 1].set_ylabel('Nombre d\'images')
axes[0, 1].axvline(df['height'].mean(), color='red', linestyle='--', label=f'Moyenne: {df["height"].mean():.0f}px')
axes[0, 1].legend()

# Scatter largeur vs hauteur
axes[1, 0].scatter(df['width'], df['height'], alpha=0.5, color='darkorange', s=15)
axes[1, 0].set_title('Largeur vs Hauteur')
axes[1, 0].set_xlabel('Largeur (px)')
axes[1, 0].set_ylabel('Hauteur (px)')
axes[1, 0].plot([0, df['width'].max()], [0, df['width'].max()], 'r--', alpha=0.4, label='Carré')
axes[1, 0].legend()

# Distribution des ratios d'aspect
axes[1, 1].hist(df['aspect_ratio'], bins=25, color='mediumpurple', edgecolor='white', alpha=0.85)
axes[1, 1].set_title('Distribution du ratio d\'aspect (L/H)')
axes[1, 1].set_xlabel('Ratio largeur/hauteur')
axes[1, 1].set_ylabel('Nombre d\'images')
axes[1, 1].axvline(1.0, color='red', linestyle='--', alpha=0.6, label='Carré (1:1)')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'resolution_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Figure sauvegardée : results/data_exploration/resolution_analysis.png")

## 4. Visualisation d'échantillons d'images

On affiche une sélection aléatoire d'images pour évaluer visuellement la **diversité des scènes** dans DIV2K.  
Cette diversité est essentielle pour entraîner un modèle de restauration qui généralise bien sur des images variées.

In [ ]:
sample_paths = random.sample(all_images, min(16, len(all_images)))

fig, axes = plt.subplots(4, 4, figsize=(16, 12))
fig.suptitle('Échantillon d\'images DIV2K — Diversité des scènes', fontsize=14, fontweight='bold')

for ax, img_path in zip(axes.flatten(), sample_paths):
    img = Image.open(img_path)
    # Réduire pour l'affichage
    img.thumbnail((300, 300))
    ax.imshow(img)
    ax.set_title(os.path.basename(img_path), fontsize=7)
    ax.axis('off')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Figure sauvegardée : results/data_exploration/sample_images.png")

## 5. Analyse des distributions de pixels et des canaux couleur

On analyse la distribution des intensités de pixels pour chaque canal RGB.  
Cette analyse permet de vérifier si les images présentent des biais de couleur ou de luminosité, ce qui pourrait nécessiter une normalisation particulière avant l'entraînement.

In [ ]:
# Analyse sur un sous-ensemble pour la rapidité
N_SAMPLE = min(50, len(all_images))
sample_for_stats = random.sample(all_images, N_SAMPLE)

r_vals, g_vals, b_vals, gray_vals = [], [], [], []
channel_means = {'R': [], 'G': [], 'B': []}

print(f"📊 Analyse des pixels sur {N_SAMPLE} images...")
for img_path in tqdm(sample_for_stats):
    img = np.array(Image.open(img_path).convert('RGB'))
    # Sous-échantillonnage pour réduire la mémoire
    step = max(1, img.shape[0] // 50)
    patch = img[::step, ::step, :]
    r_vals.extend(patch[:, :, 0].flatten().tolist())
    g_vals.extend(patch[:, :, 1].flatten().tolist())
    b_vals.extend(patch[:, :, 2].flatten().tolist())
    gray_vals.extend(np.mean(patch, axis=2).flatten().tolist())
    channel_means['R'].append(img[:, :, 0].mean())
    channel_means['G'].append(img[:, :, 1].mean())
    channel_means['B'].append(img[:, :, 2].mean())

print("✅ Analyse des pixels terminée.")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Analyse des canaux couleur et des distributions de pixels', fontsize=13, fontweight='bold')

colors = ['red', 'green', 'blue']
labels = ['Canal Rouge (R)', 'Canal Vert (G)', 'Canal Bleu (B)']
vals   = [r_vals, g_vals, b_vals]

# Histogrammes par canal
for i, (ax, c, lbl, v) in enumerate(zip(axes[0], colors, labels, vals)):
    ax.hist(v, bins=64, color=c, alpha=0.75, edgecolor='none')
    ax.set_title(lbl)
    ax.set_xlabel('Intensité (0–255)')
    ax.set_ylabel('Fréquence')
    ax.axvline(np.mean(v), color='black', linestyle='--', linewidth=1.2, label=f'μ={np.mean(v):.1f}')
    ax.legend(fontsize=9)

# Histogramme niveaux de gris
axes[1, 0].hist(gray_vals, bins=64, color='dimgray', alpha=0.75)
axes[1, 0].set_title('Distribution globale (niveaux de gris moyens)')
axes[1, 0].set_xlabel('Intensité moyenne')
axes[1, 0].set_ylabel('Fréquence')

# Moyennes par canal par image (boxplot)
axes[1, 1].boxplot([channel_means['R'], channel_means['G'], channel_means['B']],
                   labels=['R', 'G', 'B'],
                   patch_artist=True,
                   boxprops=dict(facecolor='lightyellow'),
                   medianprops=dict(color='black', linewidth=2))
axes[1, 1].set_title('Moyennes par canal (par image)')
axes[1, 1].set_ylabel('Intensité moyenne')

# Corrélation R vs B
sample_size = min(5000, len(r_vals))
idx = np.random.choice(len(r_vals), sample_size, replace=False)
axes[1, 2].scatter(np.array(r_vals)[idx], np.array(b_vals)[idx], alpha=0.05, s=2, color='purple')
axes[1, 2].set_title('Corrélation Canal R vs Canal B')
axes[1, 2].set_xlabel('Canal R')
axes[1, 2].set_ylabel('Canal B')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'pixel_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Figure sauvegardée : results/data_exploration/pixel_distribution.png")

## 6. Analyse de la taille des fichiers

La taille des fichiers reflète la complexité visuelle des images (haute fréquence, texture, contraste).  
Des fichiers volumineux indiquent des images riches en détails — ce qui est favorable pour un dataset de super-résolution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Analyse de la taille des fichiers', fontsize=13, fontweight='bold')

axes[0].hist(df['file_size_kb'], bins=40, color='teal', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribution des tailles de fichiers (KB)')
axes[0].set_xlabel('Taille (KB)')
axes[0].set_ylabel('Nombre d\'images')
axes[0].axvline(df['file_size_kb'].mean(), color='red', linestyle='--',
                label=f'Moyenne: {df["file_size_kb"].mean():.0f} KB')
axes[0].legend()

axes[1].scatter(df['pixels'] / 1e6, df['file_size_kb'], alpha=0.4, s=12, color='coral')
axes[1].set_title('Nombre de pixels vs Taille du fichier')
axes[1].set_xlabel('Mégapixels (Mpx)')
axes[1].set_ylabel('Taille (KB)')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'file_size_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Figure sauvegardée : results/data_exploration/file_size_analysis.png")

## 7. Détection d'images corrompues ou problématiques

Avant l'entraînement, il est important de vérifier l'intégrité du dataset.  
On identifie ici les images trop petites, les images dont le mode couleur est différent de RGB, et toute image ayant généré une erreur lors du chargement.

In [ ]:
# Images corrompues
if corrupted:
    print(f"❌ Images corrompues ({len(corrupted)}) :")
    for c in corrupted:
        print(f"   {c['filename']} — Erreur: {c['error']}")
else:
    print("✅ Aucune image corrompue détectée.")

# Images trop petites (moins de 200x200)
too_small = df[(df['width'] < 200) | (df['height'] < 200)]
print(f"\n⚠️  Images de petite résolution (<200px) : {len(too_small)}")

# Images non-RGB
non_rgb = df[df['mode'] != 'RGB']
print(f"⚠️  Images non-RGB : {len(non_rgb)}")
if len(non_rgb) > 0:
    print(non_rgb[['filename', 'mode']].head(10))

# Rapport final
print(f"\n📋 Bilan d'intégrité :")
print(f"   Total images     : {len(all_images)}")
print(f"   Images valides   : {len(df)}")
print(f"   Corrompues       : {len(corrupted)}")
print(f"   Trop petites     : {len(too_small)}")
print(f"   Non-RGB          : {len(non_rgb)}")
pct_ok = 100 * len(df) / len(all_images) if all_images else 0
print(f"   Taux de qualité  : {pct_ok:.1f}%")

## 8. Représentation PCA de la diversité visuelle

On applique une **Analyse en Composantes Principales (PCA)** sur des résumés statistiques de chaque image pour visualiser la **diversité** du dataset.  
Un bon dataset de restauration doit couvrir un large espace visuel : scènes naturelles, textures, structures, portraits, etc.

In [ ]:
# Extraction de features légères (stats par canal sur thumbnail)
THUMB_SIZE = 64
features = []
valid_paths = []

print(f"🔍 Extraction de features pour PCA sur {N_SAMPLE} images...")
for img_path in tqdm(sample_for_stats):
    try:
        img = np.array(Image.open(img_path).convert('RGB').resize((THUMB_SIZE, THUMB_SIZE)))
        feat = []
        for c in range(3):  # R, G, B
            ch = img[:, :, c]
            feat += [ch.mean(), ch.std(), np.percentile(ch, 25), np.percentile(ch, 75)]
        # Niveaux de gris
        gray = img.mean(axis=2)
        feat += [gray.mean(), gray.std(), float(np.mean(np.abs(np.diff(gray))))]
        features.append(feat)
        valid_paths.append(img_path)
    except Exception:
        pass

X = np.array(features)
X_scaled = StandardScaler().fit_transform(X)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Visualisation PCA
fig, ax = plt.subplots(figsize=(9, 7))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1],
                     c=np.arange(len(X_pca)), cmap='viridis',
                     s=50, alpha=0.8, edgecolors='white', linewidths=0.3)
plt.colorbar(scatter, label='Index image')
ax.set_title('PCA des images DIV2K (features statistiques RGB)', fontsize=13, fontweight='bold')
ax.set_xlabel(f'Composante 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'Composante 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'pca_diversity.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"💾 Figure sauvegardée : results/data_exploration/pca_diversity.png")
print(f"📊 Variance expliquée : PC1={pca.explained_variance_ratio_[0]*100:.1f}%, PC2={pca.explained_variance_ratio_[1]*100:.1f}%")

## 9. Récapitulatif statistique et conclusions

On génère un tableau de synthèse et on sauvegarde les métadonnées pour les notebooks suivants.

In [ ]:
# Sauvegarde du CSV de métadonnées
df.to_csv(RESULTS_DIR / 'dataset_metadata.csv', index=False)

# Tableau de synthèse
summary = {
    'Total images': len(df),
    'Largeur moyenne (px)': f"{df['width'].mean():.0f}",
    'Hauteur moyenne (px)': f"{df['height'].mean():.0f}",
    'Résolution min (px²)': f"{df['pixels'].min():,}",
    'Résolution max (px²)': f"{df['pixels'].max():,}",
    'Résolution moyenne (Mpx)': f"{df['pixels'].mean()/1e6:.2f}",
    'Taille moy. fichier (KB)': f"{df['file_size_kb'].mean():.0f}",
    'Images corrompues': len(corrupted),
    'Images non-RGB': len(non_rgb),
}

print("\n" + "="*50)
print("        RÉSUMÉ DU DATASET DIV2K")
print("="*50)
for k, v in summary.items():
    print(f"  {k:<35} : {v}")
print("="*50)
print("\n💾 Métadonnées sauvegardées : results/data_exploration/dataset_metadata.csv")

## 10. Conclusions

### 🔍 Bilan de l'exploration du dataset DIV2K

| Critère | Résultat |
|---|---|
| **Volume** | 800 images d'entraînement HR |
| **Résolution** | Images 2K, idéales pour SR |
| **Diversité** | Scènes variées (nature, villes, portraits…) |
| **Qualité** | Très peu (voire aucune) image corrompue |
| **Format** | RGB PNG, cohérent et uniforme |
| **Adéquation** | ✅ Excellente pour débruitage + super-résolution |

**Conclusions clés :**

- DIV2K est un dataset **propre, diversifié et bien adapté** aux tâches de restauration d'image.
- La haute résolution native permet de créer des **paires HR/LR** de qualité par sous-échantillonnage bicubique (×2, ×3, ×4).
- Pour le débruitage, on peut synthétiquement ajouter du bruit gaussien sur ces images HR et s'en servir comme paires noisy/clean.
- Aucun biais majeur de couleur ou de luminosité n'a été détecté — la normalisation standard `[0, 1]` sera suffisante.
- La **diversité visuelle** confirmée par la PCA garantit une bonne généralisation du modèle entraîné.

> Le dataset DIV2K est prêt pour l'entraînement. On passe au Notebook 2 — Entraînement du modèle SwinIR.